In [ ]:
#@title Install dependencies (~3 mins)
import os, time, glob, shutil, sys

model = "af2_ptm" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]
#@markdown - **model**: which set of weights to run. All of them use the same AlphaFold 3
#@markdown   graph, so everything downstream is identical. `openbind0` is OpenFold3's current
#@markdown   release and a good default; `openfold3` is their earlier preview-2, kept because
#@markdown   earlier results used it. The three `esmfold2*` entries fold from ESM-C instead
#@markdown   of an MSA -- single sequence, no search -- and differ only in the size of that
#@markdown   language model (6B, 600M, 300M). `chai1` and `esmfold2*` download and run their
#@markdown   language model automatically. `alphafold3` fetches Google DeepMind's own
#@markdown   parameters and is subject to the AF3 terms of use.

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model in your Google Drive so
#@markdown   the next session does not recompile. Measured on a 68-residue input: the first
#@markdown   prediction takes **69 s** with a cold cache and **16 s** with a warm one, so this
#@markdown   is worth about **53 s per session** (more for longer inputs). Colab wipes `/tmp`
#@markdown   between sessions, which is why it has to go somewhere else to survive. Leaving
#@markdown   it off costs only that recompile; it never changes a result.

# PINNED, both halves. Until 2026-09-16 this installed the v3.1.5 wheel for its
# compiled extension and then overlaid the Python half from the BRANCH HEAD --
# so the notebook mixed a fixed binary with a moving source tree, and two runs
# on different days could be different code. 3.1.7 is published on PyPI
# (`alphafold3-colabfold`, cp312/cp313/cp314 manylinux + macOS arm64) and its
# Python half already knows every model, so the overlay is gone and both the
# package and run_alphafold.py come from one tag.
VERSION = '3.1.7'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
IS_AF3 = (model == 'alphafold3')
# AlphaFold 2 is a SIBLING NETWORK, not one of the AF3-family ports: MSA row and
# column attention into an IPA head, reached through the same CLI and writing the
# same outputs. Its parameters are DeepMind's own release under CC BY 4.0, so they
# are fetched from source. Protein only -- a ligand or nucleotide in the input
# raises rather than folding the protein part and saying nothing.
IS_AF2 = model.startswith('af2_')
AF2_DIR = 'af2_params'
# int8 everywhere: same weights stored 8-bit and expanded on load, which is
# what keeps a Colab download to a few hundred MB. Not a knob -- there is no
# reason to pick anything else here, and AlphaFold 3's own parameters come
# from Google as float32 regardless.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # ml_collections and dm-tree ARE declared dependencies of the package, but the
  # wheel goes in with --no-deps (so the pinned jax above is not overridden), so
  # every third-party import has to be listed here. Those two are imported ONLY
  # by the af2 path (`af2/model/config.py`, and dm-tree in three more), which is
  # why every af3-family model worked and `--model af2_ptm` died with
  # `ModuleNotFoundError: No module named 'ml_collections'`.
  # The full set under src/alphafold3/af2 is: absl, haiku, jax, ml_collections,
  # numpy, scipy, tree -- the rest are already here or in Colab's base image.
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.17 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol ml_collections dm-tree")
  # aria2c, for AF2 only: its parameter tar is 5.3 GB and a single connection is
  # the bottleneck, not the link (weights._download_parallel uses it when it is
  # on PATH). Every af3-family blob is 130-350 MB, where this would not pay.
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")
  # THE FAT WHEEL, from the GitHub release -- not the slim one on PyPI.
  # `alphafold3.cpp` needs libcifpp's components.cif (518 MB raw, 120 MB
  # zipped) and cannot import without it:
  #     ImportError: Could not find the libcifpp components.cif file.
  # With the data the wheel is 130 MB, over PyPI's 100 MB per-file limit, so
  # PyPI carries the slim build (correct for anyone who provisions the data
  # themselves) and the release carries `+data`, which is self-contained.
  _whl = (f'https://github.com/sokrypton/alphafold3/releases/download/v{VERSION}'
          f'/alphafold3_colabfold-{VERSION}%2Bdata-cp313-cp313'
          f'-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl')
  os.system(f"pip install -q --no-deps '{_whl}'")
  # `run_alphafold.py` is a top-level script, not part of the package
  # (`wheel.packages = ["src/alphafold3"]`), so the wheel does not carry it.
  # Fetch it AT THE TAG so the driver and the library are the same commit.
  os.system(f'wget -q -O run_alphafold.py https://raw.githubusercontent.com'
            f'/sokrypton/alphafold3/v{VERSION}/run_alphafold.py')
  # haiku 0.0.17 still calls the moved `jax.core.DropVar`; checked against the
  # installed 0.0.17 tree, this one is still needed. (A second sed for
  # `jax.core.get_opaque_trace_state` used to sit here and never matched --
  # base.py reaches it through a `jax_core` alias and already falls back to
  # `jex_core` itself, so it was only ever a no-op.)
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, in the background. The ported models are fetched by the same code the run
# uses (alphafold3.model.weights.ensure_weights), so the run finds them already there
# and the cache layout cannot drift between the two. AlphaFold 3's own parameters are
# not ours to redistribute, so those come straight from Google.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if not os.path.isfile(STAMP):
  if IS_AF2:
    print('Downloading official AlphaFold 2 parameters (CC BY 4.0)...')
    with open('prefetch_af2.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_af2_params(sys.argv[1]))\n')
    os.system(f'(python prefetch_af2.py {AF2_DIR} && touch {STAMP}) &')
  elif IS_AF3:
    print("Downloading official AlphaFold 3 weights (public, no login required)...")
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -q -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}" && touch {STAMP}) &')
  else:
    print(f'Downloading {model} weights...')
    with open('prefetch_weights.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n')
    os.system(f'(python prefetch_weights.py {model} {PRECISION} && touch {STAMP}) &')

# Where the compiled model is cached. /tmp is wiped when the VM goes away, so a
# fresh session recompiles (~53 s on a small input); Drive survives. Opt-in, and
# the run falls back to /tmp if the mount does not work rather than failing.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

for sentinel in (STAMP, 'DATA_DONE'):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')



In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-c',
  'from alphafold3.af2 import inference;'
  'from alphafold3.af2.model import config, data, model;'
  'import ml_collections, tree;'
  'print("AF2 IMPORT OK")'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1500:] if r.returncode else '')
import shutil
print('aria2c on PATH:', bool(shutil.which('aria2c')))
print('SMOKE_RESULT:', 'PASS' if r.returncode == 0 else 'FAIL rc=%d' % r.returncode)
